## Setup

In [1]:
!git clone --branch 1-ddpm-implementation https://github.com/GoharKhachatryan/GenAI_project.git

Cloning into 'GenAI_project'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 71 (delta 29), reused 47 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 296.36 KiB | 2.65 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [2]:
%cd GenAI_project

/content/GenAI_project


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ln -s /content/drive/MyDrive/for_bostongene/data /content/GenAI_project/data
!ln -s /content/drive/MyDrive/for_bostongene/dataset.pt /content/GenAI_project/dataset.pt

In [5]:
%cd /content/GenAI_project/

/content/GenAI_project


## Conditional DDPM training

I have decided to implement the training loop in a notebook for a better visualization and use.

This notebook trains a conditional DDPM on the paired CIFAR-10 dataset.

The denoising model predicts the Gaussian noise added to an image at
a randomly sampled diffusion timestep. The model is conditioned on a
16-dimensional vector derived deterministically from the corresponding
clean image.

A cosine noise schedule with T=1000 diffusion steps is used.

## Imports

In [6]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

from prepare_dataset import PairedCIFAR10

from scripts.unet import UNet
from scripts.model import ConditionalDenoiser
from scripts.diffusion import DDPM

In [7]:
# In case you want to reproduce the training process
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Device selection

In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


## Load datasets

In [9]:
train_dataset = PairedCIFAR10("train")
val_dataset = PairedCIFAR10("val")
test_dataset = PairedCIFAR10("test")

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset),
)

45000 5000 10000


## Condition Normalization

In [10]:
# Based on the condition analysis, they need to be normalized
# feature-wise before feeding to the model
train_conds = torch.stack([
    train_dataset[i][1]
    for i in range(len(train_dataset))
])

# To prevent data leakage, use the statistics from the training set,
# to normalize all of the sets
cond_mean = train_conds.mean(dim=0)
cond_std = train_conds.std(dim=0)

print("Condition mean:")
print(cond_mean)

print("\nCondition std:")
print(cond_std)

Condition mean:
tensor([ 1.6590e+01,  1.5525e+01,  1.4583e+01,  1.4828e+01,  1.5473e+01,
         1.5303e+01,  1.5303e+01,  1.5448e+01,  4.9124e-01,  4.8195e-01,
         4.4636e-01,  1.9618e-01, -5.2558e-09, -3.2213e-09, -3.7723e-09,
         2.3736e-09])

Condition std:
tensor([6.6494, 4.4762, 4.0474, 5.3317, 5.0976, 3.8374, 3.8600, 5.0750, 0.1283,
        0.1257, 0.1531, 0.0601, 1.0000, 1.0000, 1.0000, 1.0000])


In [11]:
def normalize_condition(cond):
    return (
        cond - cond_mean.to(cond.device)
    ) / (
        cond_std.to(cond.device) + 1e-8
    )

In [12]:
# Save these statistics, since they will be used for the generation
torch.save(
    {
        "mean": cond_mean,
        "std": cond_std,
    },
    "cond_stats.pt",
)

## DataLoaders

In [14]:
# For a T4
# In case of OOM, change into 64 or lower
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

## Defininig the full model

In [18]:
# Creating the UNet model...
unet = UNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    emb_dim=256,
)

# ...and the conditional denoiser
model = ConditionalDenoiser(
    unet=unet,
    time_dim=256,
    cond_dim=16,
    emb_dim=256,
)

model = model.to(device)

In [19]:
# Since the task has a given range of parameters (3 - 10M parameters),
# checking here if my model is within the given range
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total trainable parameters: "
    f"{num_params / 1e6:.2f}M"
)

Total trainable parameters: 9.22M


In [20]:
# Manually implemented DDPM
ddpm = DDPM(
    timesteps=1000,
    schedule="cosine",
    device=device,
)

In [21]:
# A basic optimizer for now
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4,
)

## Sanity check

In [ ]:
# take one batch
images, cond, _ = next(
    iter(train_loader)
)

images = images.to(device)
cond = normalize_condition(
    cond.to(device)
)

In [ ]:
test_losses = []

model.train()

for step in range(200):

    optimizer.zero_grad()

    t = torch.randint(
        0,
        ddpm.timesteps,
        (images.shape[0],),
        device=device,
        dtype=torch.long,
    )

    loss = ddpm.training_loss(
        model,
        images,
        t,
        cond,
    )

    loss.backward()
    optimizer.step()

    test_losses.append(
        loss.item()
    )

    if step % 20 == 0:
        print(
            f"Step {step:03d} | "
            f"Loss {loss.item():.4f}"
        )

In [ ]:
plt.plot(test_losses)
plt.xlabel("Optimization step")
plt.ylabel("Noise prediction MSE")
plt.title("Single-batch overfitting sanity check")
plt.show()